In [1]:
import duckdb

con = duckdb.connect("../data/intermediate/gdelt.db")

In [3]:
con.execute("SHOW TABLES").fetchdf()

,name
0,country_month_features
1,exports
2,exports_clean
3,incoming_country_month
4,narrative_country_month
5,narrative_country_month_pilot
6,narrative_nvi_pilot
7,narrative_volatility_filtered
8,narrative_volatility_final
9,narrative_volatility_pilot


In [4]:
con.execute("""SELECT COUNT(*) FROM exports;""").fetchdf()

,count_star()
0,7794077


In [25]:
import pandas as pd

cols = con.execute("DESCRIBE exports").fetchdf()["column_name"].tolist()
results = []

for c in cols:
    res = con.execute(f"""
        SELECT
            '{c}' AS column_name,
            COUNT(*) - COUNT({c}) AS null_count,
            COUNT(*) AS total_rows,
            ROUND((COUNT(*) - COUNT({c})) * 100.0 / COUNT(*), 2) AS null_percentage
        FROM exports
    """).fetchdf()
    results.append(res)

# DataFrame olarak birleştir
null_df = pd.concat(results, ignore_index=True)

# Null oranına göre sırala
null_df.sort_values("null_percentage", ascending=False)

,column_name,null_count,total_rows,null_percentage
24,Actor2Type3Code,7791678,7794077,99.97
14,Actor1Type3Code,7790419,7794077,99.95
21,Actor2Religion2Code,7776823,7794077,99.78
11,Actor1Religion2Code,7771538,7794077,99.71
19,Actor2EthnicCode,7759256,7794077,99.55
...,...,...,...,...
30,GoldsteinScale,25,7794077,0.00
32,NumSources,0,7794077,0.00
51,ActionGeo_Type,0,7794077,0.00
59,DATEADDED,0,7794077,0.00


In [29]:
# Null olan kolonları filtrele
null_cols = null_df[null_df["null_count"] > 0]

# Kaç kolon null içeriyor
num_null_cols = null_cols.shape[0]

# Toplam null sayısı (tüm kolonlar için)
total_nulls = null_cols["null_count"].sum()

print(f"Null içeren kolon sayısı: {num_null_cols}")
print(f"Tüm null değerlerin toplam sayısı: {total_nulls}")

Null içeren kolon sayısı: 42
Tüm null değerlerin toplam sayısı: 149562341


In [30]:
# %90'dan fazla null olan kolonlar
high_null_cols = null_df[null_df["null_percentage"] > 90]["column_name"].tolist()

# clean kolonları seç
cols_to_keep = [c for c in cols if c not in high_null_cols]
cols_sql = ", ".join(cols_to_keep)

# clean tabloyu DB'de oluştur
con.execute(f"""
CREATE OR REPLACE TABLE exports_clean AS
SELECT {cols_sql}
FROM exports
""")

In [32]:
cols_clean = con.execute("DESCRIBE exports_clean").fetchdf()["column_name"].tolist()
results_clean = []

for c in cols_clean:
    res = con.execute(f"""
        SELECT
            '{c}' AS column_name,
            COUNT(*) - COUNT({c}) AS null_count,
            COUNT(*) AS total_rows,
            ROUND((COUNT(*) - COUNT({c})) * 100.0 / COUNT(*), 2) AS null_percentage
        FROM exports_clean
    """).fetchdf()
    results_clean.append(res)

null_clean_df = pd.concat(results_clean, ignore_index=True)
null_clean_df.sort_values("null_percentage", ascending=False)

,column_name,null_count,total_rows,null_percentage
12,Actor2Type1Code,5201230,7794077,66.73
35,Actor2Geo_ADM2Code,4947917,7794077,63.48
8,Actor1Type1Code,4429329,7794077,56.83
11,Actor2CountryCode,4293556,7794077,55.09
43,ActionGeo_ADM2Code,3787930,7794077,48.60
27,Actor1Geo_ADM2Code,3665521,7794077,47.03
7,Actor1CountryCode,3355779,7794077,43.06
32,Actor2Geo_FullName,2396560,7794077,30.75
36,Actor2Geo_Lat,2396563,7794077,30.75
37,Actor2Geo_Long,2395828,7794077,30.74


In [4]:
cols_info = con.execute("DESCRIBE exports_clean").fetchdf()
cols_info

,column_name,column_type,null,key,default,extra
0,GlobalEventID,BIGINT,YES,None,None,None
1,Day,BIGINT,YES,None,None,None
2,MonthYear,BIGINT,YES,None,None,None
3,Year,BIGINT,YES,None,None,None
4,FractionDate,DOUBLE,YES,None,None,None
5,Actor1Code,VARCHAR,YES,None,None,None
6,Actor1Name,VARCHAR,YES,None,None,None
7,Actor1CountryCode,VARCHAR,YES,None,None,None
8,Actor1Type1Code,VARCHAR,YES,None,None,None
9,Actor2Code,VARCHAR,YES,None,None,None


In [33]:
con.execute("""
SELECT Actor1CountryCode, COUNT(*) AS count
FROM exports_clean
GROUP BY Actor1CountryCode
ORDER BY count DESC
LIMIT 50;
            """).fetchdf()

,Actor1CountryCode,count
0,Unknown,3355779
1,USA,1378334
2,GBR,226721
3,ISR,137760
4,RUS,137600
5,CHN,134988
6,CAN,120503
7,AUS,113043
8,NGA,105154
9,IND,101248


In [34]:
cols = [
"Actor1CountryCode",
"Actor2CountryCode",
"Actor1Geo_CountryCode",
"Actor2Geo_CountryCode",
"ActionGeo_CountryCode"
]

for c in cols:
    con.execute(f"""
        UPDATE exports_clean
        SET {c} = 'Unknown'
        WHERE {c} IS NULL
    """)

In [35]:
con.execute("""
SELECT ActionGeo_CountryCode, COUNT(*)
FROM exports_clean
GROUP BY ActionGeo_CountryCode
ORDER BY COUNT(*) DESC
""").fetchdf()

,ActionGeo_CountryCode,count_star()
0,US,2494819
1,IN,412272
2,UK,398751
3,IS,260715
4,NI,242610
...,...,...
250,WQ,4
251,JQ,4
252,JN,3
253,IP,2


In [5]:
con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT GlobalEventID) AS unique_events
FROM exports_clean
""").fetchdf()

,total_rows,unique_events
0,7794077,7794077


In [6]:
con.execute("""
SELECT
MIN(GoldsteinScale),
MAX(GoldsteinScale),
MIN(AvgTone),
MAX(AvgTone),
MIN(NumMentions),
MAX(NumMentions)
FROM exports_clean
""").fetchdf()

,min(GoldsteinScale),max(GoldsteinScale),min(AvgTone),max(AvgTone),min(NumMentions),max(NumMentions)
0,-10.0,10.0,-39.393939,32.407407,1,1020


In [7]:
con.execute("""
SELECT QuadClass, COUNT(*) AS count
FROM exports_clean
GROUP BY QuadClass
ORDER BY count DESC
""").fetchdf()

,QuadClass,count
0,1,4746039
1,4,1089910
2,3,1061948
3,2,896180


In [8]:
con.execute("""
SELECT EventRootCode, COUNT(*) AS count
FROM exports_clean
GROUP BY EventRootCode
ORDER BY count DESC
LIMIT 20
""").fetchdf()

,EventRootCode,count
0,04,1861966
1,01,1149065
2,05,694434
3,02,552462
4,11,521264
5,03,488112
6,19,472762
7,17,391872
8,08,282452
9,07,269426


In [3]:
con.execute("""
SELECT COUNT(*)
FROM exports_clean
WHERE ActionGeo_Lat = 0 AND ActionGeo_Long = 0
""").fetchdf()



,count_star()
0,0
